# Проект 1. Каналы привлечения ПРАЙМ: какой эффективнее и какой теряет больше клиентов

**Бизнес-вопрос.** Какой канал привлечения даёт больше всего выручки на рубль затрат, а какой теряет больше всего клиентов? От ответа зависит, куда направлять маркетинговый бюджет.

**Определения**
- **Эффективность канала** - выручка за 14 месяцев на зарегистрированного клиента / CAC и срок окупаемости (месяц жизни, когда накопленная выручка догнала CAC; месяц жизни = 30 дней).
- **Потери канала** - доля зарегистрированных, дошедших до 6-й оплаты, и доля потерянных между 2-й и 6-й оплатой.

**Гипотезы**
1. Каналы различаются не входом (подпиской), а удержанием.
2. Удержание — главная причина разницы в эффективности каналов.

## 0. Подключение


In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sqlalchemy import text

from prime.config import get_engine

sns.set_theme(style="whitegrid", context="notebook")
GREY, ACC = "#B8BCC4", "#D1492E"  # серый для всех, акцент для главного

engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            # Сутки и месяцы считаем по UTC — так они у всех одинаковые.
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


print("подключились как:", q("select current_user as login").iloc[0, 0])


## 1. Как выбирал вопрос

Сначала посмотрел на общую динамику выручки, отмены по причинам и качество данных:
- **дубли списаний** (повтор по той же подписке и сумме в пределах 20 минут) — около 0,1 % выручки в каждом месяце, на выводы не влияют;
- **мартовское подорожание 2026** (Базовый 199 → 249, Расширенный 399 → 449 ₽) дало +18 % к среднему чеку; гипотеза «подорожание выгнало клиентов» на моём сегменте в ДЗ-2 не подтвердилась;
- **дыра в платежах 14–15 апреля 2026** — сравнивать месяцы можно только в пересчёте на день;
- **август 2026** неполный (данные до 30-го), плюс массовые отмены `partner_telecom` 22–23 августа.

Динамика выручки ровная, а самые большие различия видны между каналами, поэтому дальше исследую каналы привлечения.


In [ ]:
monthly_rev = q("""
    with p as (
        select payment_id, subscription_id, paid_at, amount, status,
               paid_at - lag(paid_at) over (
                   partition by subscription_id, amount order by paid_at
               ) as gap
        from prime.payments
    ),
    flagged as (
        select *, (gap < interval '20 minutes') as is_dup
        from p
    )
    select to_char(date_trunc('month', paid_at), 'YYYY-MM') as month,
           count(*) filter (where status = 'success')                      as payments_raw,
           count(*) filter (where status = 'success' and coalesce(is_dup, false)) as dup_payments,
           sum(amount) filter (where status = 'success')::float8           as revenue_raw,
           sum(amount) filter (where status = 'success'
                               and not coalesce(is_dup, false))::float8    as revenue_clean
    from flagged
    group by 1
    order by 1
""")

# Январь 2025 (старт с 6-го) и август 2026 (до 30-го) неполные
monthly_rev["full_month"] = ~monthly_rev["month"].isin(["2025-01", "2026-08"])
monthly_rev["growth_pct"] = monthly_rev["revenue_clean"].pct_change() * 100
monthly_rev.round(1)


In [ ]:
cancels = q("""
    select to_char(date_trunc('month', ended_at), 'YYYY-MM') as month,
           cancel_reason,
           count(*) as n
    from prime.subscriptions
    where ended_at >= '2025-11-01' and ended_at < '2026-09-01'
    group by 1, 2
    order by 1, 2
""")
cancels.pivot(index="month", columns="cancel_reason", values="n")

In [ ]:
daily = q("""
    select paid_at::date as day, count(*) as n, sum(amount)::float8 as revenue
    from prime.payments
    where status = 'success'
      and paid_at >= '2026-04-08' and paid_at < '2026-04-23'
    group by 1 order by 1
""")
daily

## 2. Зерно, когорта, период

- **Зерно - клиент.** CAC считается на клиента, значит и выручку надо считать на того же клиента: знаменатель одинаковый.
- **Когорта - регистрация 01.02–30.06.2025.** У всех клиентов есть минимум 14 месяцев истории, поэтому удержание сравнимо. Более поздние когорты ещё не «дозрели».
- **Выручка - только `status = 'success'`.** Платёж связан с подпиской напрямую, `subscription_members` не используется, поэтому двойного счёта из-за шеринга нет.
- **Клиенты без канала** остаются отдельной строкой «без канала» в воронке и выручке, но без CAC: расходов на них в `marketing_spend` нет.
- **Проверка соединения:** `rows_after_join == registered` в каждом канале, то есть у клиента не больше одной подписки и воронка не задвоилась.


## 3. Воронка по каналам


In [ ]:
funnel = q("""
with cohort as (
    select client_id,
           coalesce(acquisition_channel, 'без канала') as channel,
           registered_at
    from prime.clients
    where registered_at >= '2025-02-01' and registered_at < '2025-07-01'
),
paid as (
    select s.subscription_id, count(*) as n_paid
    from prime.subscriptions s
    join prime.payments p on p.subscription_id = s.subscription_id
    where p.status = 'success'
      and (s.ended_at is null or p.paid_at <= s.ended_at)
    group by 1
)
select c.channel,
       count(*)                                         as rows_after_join,
       count(distinct c.client_id)                      as registered,
       count(s.subscription_id)                         as subscribed,
       count(*) filter (where s.started_at < c.registered_at + interval '30 days') as subscribed_30d,
       count(*) filter (where paid.n_paid >= 1)         as paid_1,
       count(*) filter (where paid.n_paid >= 2)         as paid_2,
       count(*) filter (where paid.n_paid >= 6)         as paid_6
from cohort c
left join prime.subscriptions s on s.owner_client_id = c.client_id
left join paid on paid.subscription_id = s.subscription_id
group by 1
order by 1
""")
funnel

**Результат.** Подписываются 69–70 % клиентов в любом канале, за 30 дней около 34 %: вход одинаковый. Каналы расходятся на удержании: до 6-й оплаты доходят 61,7 % клиентов `organic` и только 47,1 % клиентов `social`. **Гипотеза 1 подтвердилась.**


In [ ]:
f = funnel.set_index("channel")
f["share_6"] = f["paid_6"] / f["registered"] * 100
f = f.sort_values("share_6")

fig, ax = plt.subplots(figsize=(8, 4))
colors = [ACC if ch == "social" else GREY for ch in f.index]
ax.barh(f.index, f["share_6"], color=colors)
for y, v in enumerate(f["share_6"]):
    ax.text(v + 0.6, y, f"{v:.1f}%", va="center")
ax.set_xlabel("Доля зарегистрированных, дошедших до 6-й оплаты, %")
ax.set_ylabel("")
ax.set_title("Каналы расходятся на удержании: social теряет больше всех")
ax.set_xlim(0, 70)
plt.tight_layout(); plt.show()

## 4. CAC, выручка на клиента и окупаемость

CAC = расходы канала за период когорты / зарегистрированные клиенты канала. Выручка на клиента — накопленная выручка когорты по месяцам жизни / зарегистрированные.


In [ ]:
# 1. Расходы на привлечение за период когорты
spend_g = q("""
    select channel, sum(amount)::float8 as spent
    from prime.marketing_spend
    where spend_date >= '2025-02-01' and spend_date < '2025-07-01'
    group by 1 order by 1
""")

# 2. Выручка когорты по месяцу жизни клиента (агрегат в базе, строки не тянем)
life = q("""
    select coalesce(c.acquisition_channel, 'без канала') as channel,
           extract(day from p.paid_at - c.registered_at)::int / 30 + 1 as life_month,
           sum(p.amount)::float8 as revenue
    from prime.payments p
    join prime.subscriptions s on s.subscription_id = p.subscription_id
    join prime.clients c on c.client_id = s.owner_client_id
    where c.registered_at >= '2025-02-01' and c.registered_at < '2025-07-01'
      and p.status = 'success'
    group by 1, 2
    order by 1, 2
""")

# 3. Зарегистрированные из воронки выше, CAC на зарегистрированного
registered = funnel.set_index("channel")["registered"]
cac = spend_g.set_index("channel")["spent"] / registered
print(cac.round(0).sort_values())

# 4. Накопленная выручка на зарегистрированного, до 14-го месяца
by_month = (life[life["life_month"] <= 14]
            .pivot_table(index="life_month", columns="channel",
                         values="revenue", aggfunc="sum", fill_value=0))
curve = by_month.cumsum() / registered

# 5. Срок окупаемости: первый месяц, когда накопленное >= CAC.
# Без CAC у «без канала» и пропуск даст NaN, это нормально.
hit = curve.ge(cac, axis=1)
payback = hit.idxmax().where(hit.any())

pd.DataFrame({
    "cac": cac.round(0),
    "revenue_14m": curve.loc[14].round(0),
    "ratio_14m_cac": (curve.loc[14] / cac).round(2),
    "payback_month": payback,
}).sort_values("ratio_14m_cac")

In [ ]:
rev14 = curve.loc[14]                       # накопленная выручка на клиента за 14 мес.
summary = pd.DataFrame({
    "cac": cac,
    "revenue_14m": rev14,
    "ratio_14m_cac": rev14 / cac,
    "payback_month": payback,
})
summary

In [ ]:
order = summary.dropna(subset=["cac"]).sort_values("ratio_14m_cac", ascending=False).index

fig, axes = plt.subplots(2, 3, figsize=(12, 6), sharex=True, sharey=True)
for ax, ch in zip(axes.flat, order):
    hot = ch == "performance"
    ax.plot(curve.index, curve[ch], color=ACC if hot else "#555B66", lw=2)
    ax.axhline(cac[ch], color=ACC if hot else GREY, ls="--", lw=1.5)
    pb = payback[ch]
    ax.scatter([pb], [curve.loc[pb, ch]], color=ACC if hot else "#555B66", zorder=3)
    ax.set_title(f"{ch}: окупается на {int(pb)}-м мес.")
    ax.text(14, cac[ch], " CAC", va="bottom", ha="right", fontsize=9)
for ax in axes[1]:
    ax.set_xlabel("Месяц жизни клиента")
for ax in axes[:, 0]:
    ax.set_ylabel("Накопленная выручка\nна клиента, ₽")
fig.suptitle("Выручка на клиента почти одинакова, а CAC различается в 10 раз", y=1.0)
plt.tight_layout(); plt.show()

In [ ]:
s = summary.dropna(subset=["cac"]).sort_values("ratio_14m_cac")

fig, ax = plt.subplots(figsize=(8, 4))
colors = [ACC if ch == "performance" else GREY for ch in s.index]
ax.barh(s.index, s["ratio_14m_cac"], color=colors)
for y, v in enumerate(s["ratio_14m_cac"]):
    ax.text(v + 0.2, y, f"{v:.1f}×", va="center")
ax.axvline(1, color="black", lw=1)
ax.text(1.1, -0.45, "порог 1×: выручка = CAC", fontsize=9)
ax.set_xlabel("Выручка за 14 месяцев / CAC (выручка, не маржа)")
ax.set_ylabel("")
ax.set_title("Performance почти не окупается: 1,25× за 14 месяцев")
plt.tight_layout(); plt.show()

**Результат.** `organic` окупается за 2 месяца (15,5× за 14 месяцев), `performance` — только на 11-м месяце (1,25×). Выручка на клиента у каналов близка (1 711–2 176 ₽), а CAC различается в 10,5 раза. Разрыв `organic` и `performance` (12,4×) на 93 % объясняется CAC и только на 7 % выручкой на клиента (расчёт ниже). **Гипотеза 2 не подтвердилась**: удержание различается, но эффективность определяет цена привлечения.


## 5. Факторная декомпозиция: изменение среднего CAC, 02–06.2025 → 02–06.2026

Средний CAC = Σ (доля канала в регистрациях × CAC канала). Изменение раскладывается на:
- **эффект структуры** — изменились доли каналов (при старых CAC);
- **эффект ставки** — изменился CAC внутри канала (при новых долях).

Сумма эффектов равна изменению — это проверяется ниже. Клиенты без канала исключены из обоих периодов.


In [ ]:
per = q("""
with sp as (
  select channel,
         case when spend_date >= '2025-02-01' and spend_date < '2025-07-01' then 'p1'
              when spend_date >= '2026-02-01' and spend_date < '2026-07-01' then 'p2' end as period,
         sum(amount)::float8 as spent
  from prime.marketing_spend
  group by 1, 2
),
rg as (
  select coalesce(acquisition_channel, 'без канала') as channel,
         case when registered_at >= '2025-02-01' and registered_at < '2025-07-01' then 'p1'
              when registered_at >= '2026-02-01' and registered_at < '2026-07-01' then 'p2' end as period,
         count(*) as registered
  from prime.clients
  group by 1, 2
)
select rg.channel, rg.period, rg.registered, sp.spent
from rg left join sp on sp.channel = rg.channel and sp.period = rg.period
where rg.period is not null
order by 1, 2
""")

d = per[per.channel != "без канала"].pivot(index="channel", columns="period",
                                          values=["registered", "spent"])
reg1, reg2 = d["registered"]["p1"], d["registered"]["p2"]
sp1, sp2 = d["spent"]["p1"], d["spent"]["p2"]

w1, w2 = reg1 / reg1.sum(), reg2 / reg2.sum()   # доли в регистрациях
cac1, cac2 = sp1 / reg1, sp2 / reg2             # CAC внутри канала

mix  = (w2 - w1) * cac1      # структура: смена микса при старых ценах
rate = w2 * (cac2 - cac1)    # ставка: изменение CAC внутри канала при новом миксе
total = sp2.sum() / reg2.sum() - sp1.sum() / reg1.sum()

res = pd.DataFrame({"w1": w1, "w2": w2, "cac1": cac1, "cac2": cac2, "mix": mix, "rate": rate})
print("сумма эффектов:", round((mix + rate).sum(), 2), "| изменение CAC:", round(total, 2))
res.round(3)

In [ ]:
r = res.copy()
r["total"] = r["mix"] + r["rate"]
r = r.sort_values("total")

fig, ax = plt.subplots(figsize=(8, 4))
y = np.arange(len(r))
ax.barh(y - 0.2, r["mix"],  height=0.4, color=GREY, label="Структура (микс каналов)")
ax.barh(y + 0.2, r["rate"], height=0.4, color=ACC,  label="Ставка (CAC внутри канала)")
ax.set_yticks(y); ax.set_yticklabels(r.index)
ax.axvline(0, color="black", lw=1)
ax.set_xlabel("Вклад в изменение среднего CAC, ₽ на клиента")
ax.set_title(f"CAC вырос на {r['total'].sum():.1f} ₽ (2025 → 2026): почти всё даёт performance")
ax.legend(loc="lower right")
plt.tight_layout(); plt.show()

**Результат.** Средний CAC вырос на 13,5 ₽ (около 2 %). Эффект ставки +10,6 ₽ (78 %): почти всё дают `performance` (+10,9 ₽) и `social` (+3,9 ₽), остальные каналы подешевели. Эффект структуры +3,0 ₽ (22 %). **Вывод:** дорожают именно те каналы, которые и так самые дорогие и хуже всех удерживают.


### Проверки


In [ ]:
# 1. Декомпозиция сходится
assert abs((res.mix + res.rate).sum() - (res.w2 * res.cac2).sum() + (res.w1 * res.cac1).sum()) < 1e-6

# 2. Воронка не задвоилась при соединении
assert (funnel["rows_after_join"] == funnel["registered"]).all()

# 3. Цена против выручки: organic vs performance
a, b = "organic", "performance"
d_cac = np.log(cac[b] / cac[a])
d_rev = np.log(rev14[a] / rev14[b])
total = d_cac + d_rev
print(f"ratio {summary.loc[a,'ratio_14m_cac']/summary.loc[b,'ratio_14m_cac']:.1f}×; "
      f"на CAC {d_cac/total:.0%}, на выручку {d_rev/total:.0%}")

## 6. Вывод и рекомендация

- **Самый эффективный канал - `organic`** (окупается за 2 месяца), среди платных — `crm` и `referral` (3 месяца).
- **Больше всего клиентов теряет `social`** (до 6-й оплаты доходят 47 %), **больше всего денег - `performance`** (1,25× за 14 месяцев, окупаемость на 11-м месяце, и его CAC продолжает расти).

**Рекомендации**
1. Провести эксперимент: сократить бюджет `performance` в части регионов и измерить, как изменятся регистрации и предельный CAC.
2. Для `social` и `performance` добавить меры удержания после 2-й оплаты.
3. Проверить, масштабируются ли `crm` и `referral`: они дёшевы и удерживают, но их объём ограничен.

**Ограничения**
- Считается выручка, а не маржа: выплат партнёрам и кэшбэка в расчёте нет. Если маржа ниже ~80 %, `performance` за 14 месяцев не окупается.
- Средний CAC не равен предельному: дешёвый канал может не выдержать роста бюджета.
